## Export Deeplabcut Model to .onnx

In [5]:
import torch
from deeplabcut.pose_estimation_pytorch.models.model import PoseModel
import yaml

### Fill fields and run all

In [6]:
model = "kneeap"
# model = "cpak"

shuffle = 3

# pytorch_config.yaml
pytorch_config_path = f"/Users/furkan/projects/cpak-prediction/notebooks/models/{model}-sh{shuffle}-ep1000/pytorch_config.yaml"

# best.pt or any .pt
pt_path = f"/Users/furkan/projects/cpak-prediction/notebooks/models/{model}-sh{shuffle}-ep1000/snapshot-1000.pt"

# output file name
onnx_file_path = f"out/models/{model}-sh{shuffle}-ep1000.onnx"

In [7]:
with open(pytorch_config_path) as f:
    config = yaml.safe_load(f)

model = PoseModel.build(config["model"]).to("mps")
model.eval()

PoseModel(
  (backbone): ResNet(
    (model): ResNet(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): GroupNorm(32, 64, eps=1e-05, affine=True)
      (act1): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): GroupNorm(32, 64, eps=1e-05, affine=True)
          (act1): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): GroupNorm(32, 64, eps=1e-05, affine=True)
          (drop_block): Identity()
          (act2): ReLU(inplace=True)
          (aa): Identity()
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): GroupNorm(32, 256, eps=1e-05, affine=True)
          (act3): ReLU(inplace=True)
          

In [8]:
snapshot = torch.load(pt_path, map_location="mps")
model.load_state_dict(snapshot["model"])

/var/folders/4j/vmfcbk011r1d4ztywrf3d_rh0000gn/T/ipykernel_60133/1540561732.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  snapshot = torch.load(pt_path, map_location="

<All keys matched successfully>

In [9]:
# 1. Modeli test moduna aldığından emin ol
model.eval()

dummy_input = torch.randn(1, 3, 640, 1536).to("mps")

print("export starting..")

# 4. Dışa Aktarma (Export) işlemi
torch.onnx.export(
    model,                      # Yüklediğimiz ve ağırlıklarını eşleştirdiğimiz model
    dummy_input,                # Modelin yapısını çözmesi için gereken sahte girdi
    onnx_file_path,             # Kaydedilecek dosya adı
    export_params=True,         # Ağırlıkları dosyanın içine göm (En önemlisi!)
    opset_version=11,           # Çoğu platformla uyumlu, kararlı ONNX sürümü
    do_constant_folding=True,   # Çıkarım hızını artırmak için matematiksel optimizasyonlar yapar
    input_names=['input'],      # Girdi katmanının adı
    output_names=['output'],    # Çıktı katmanının adı
    
    # DİNAMİK EKSENLER: Bu ayar sayesinde model sadece 480x640 resimlere sıkışıp kalmaz.
    # İstediğin boyutta (veya batch sayısında) resim verebilirsin.
    dynamic_axes={             
        'input': {0: 'batch_size', 2: 'height', 3: 'width'},
        'output': {0: 'batch_size', 2: 'height', 3: 'width'}
    }
)

print(f"{onnx_file_path} successfully created. ")

export starting..
out/models/kneeap-sh3-ep1000.onnx successfully created. 
